# Tile-based CNN training (256x256)


In [1]:
from pathlib import Path
import zarr
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path("/home/naren-root/Documents/FYP2/Project")
ROOT_30M = PROJECT_ROOT / "madurai_30m.zarr"
ROOT_DAILY = PROJECT_ROOT / "madurai.zarr"
COMMON_DATES = PROJECT_ROOT / "common_dates.csv"

root_30m = zarr.open_group(str(ROOT_30M), mode="r")
root_daily = zarr.open_group(str(ROOT_DAILY), mode="r")

def _to_str(arr):
    return np.array([x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else str(x) for x in arr])

common_df = pd.read_csv(COMMON_DATES)
common_dates = pd.to_datetime(common_df["landsat_date"], errors="coerce").dropna()
common_dates = pd.DatetimeIndex(common_dates).sort_values()

daily_raw = _to_str(root_30m["time"]["daily"][:])
monthly_raw = _to_str(root_30m["time"]["monthly"][:])

daily_times = pd.to_datetime(daily_raw, format="%Y_%m_%d", errors="coerce").dropna()
monthly_times = pd.to_datetime(monthly_raw, format="%Y_%m", errors="coerce").dropna()

daily_idx = np.flatnonzero(daily_times.isin(common_dates))
month_index = pd.DatetimeIndex(common_dates.to_period("M").to_timestamp())
monthly_idx = np.flatnonzero(monthly_times.isin(month_index))

# map each daily date to its monthly index
monthly_map = {t: i for i, t in enumerate(monthly_times)}
daily_to_month = []
for t in daily_times[daily_idx]:
    m = t.to_period("M").to_timestamp()
    daily_to_month.append(monthly_map.get(m, -1))
daily_to_month = np.array(daily_to_month)
valid = daily_to_month >= 0
daily_idx = daily_idx[valid]
daily_to_month = daily_to_month[valid]

print(f"daily_idx={len(daily_idx)} monthly_idx={len(monthly_idx)}")

# reference high-res shape (landsat)
landsat_shape = root_30m["labels_30m"]["landsat"]["data"].shape
H_hr, W_hr = landsat_shape[-2], landsat_shape[-1]

# low-res shapes (modis/viirs)
modis_shape = root_daily["products"]["modis"]["data"].shape
viirs_shape = root_daily["products"]["viirs"]["data"].shape
H_lr_modis, W_lr_modis = modis_shape[-2], modis_shape[-1]
H_lr_viirs, W_lr_viirs = viirs_shape[-2], viirs_shape[-1]

# precompute index maps for nearest-neighbor upsampling into ROI grid
y_map_modis = np.floor(np.linspace(0, H_lr_modis - 1, H_hr)).astype(np.int64)
x_map_modis = np.floor(np.linspace(0, W_lr_modis - 1, W_hr)).astype(np.int64)
y_map_viirs = np.floor(np.linspace(0, H_lr_viirs - 1, H_hr)).astype(np.int64)
x_map_viirs = np.floor(np.linspace(0, W_lr_viirs - 1, W_hr)).astype(np.int64)

patch_size = 256


daily_idx=261 monthly_idx=96


In [2]:
class TileDataset(Dataset):
    def __init__(self, n_samples, seed=0):
        self.n_samples = n_samples
        self.rng = np.random.default_rng(seed)

        self.g_era5 = root_30m["products_30m"]["era5"]["data"]
        self.g_landsat = root_30m["labels_30m"]["landsat"]["data"]
        self.g_s1 = root_30m["products_30m"]["sentinel1"]["data"]
        self.g_s2 = root_30m["products_30m"]["sentinel2"]["data"]
        self.g_modis = root_daily["products"]["modis"]["data"]
        self.g_viirs = root_daily["products"]["viirs"]["data"]

        self.g_dem = root_30m["static_30m"]["dem"]["data"]
        self.g_world = root_30m["static_30m"]["worldcover"]["data"]
        self.g_dyn = root_30m["static_30m"]["dynamic_world"]["data"]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        i = self.rng.integers(0, len(daily_idx))
        t = int(daily_idx[i])
        m = int(daily_to_month[i])

        y0 = self.rng.integers(0, H_hr - patch_size + 1)
        x0 = self.rng.integers(0, W_hr - patch_size + 1)
        y1 = y0 + patch_size
        x1 = x0 + patch_size

        # high-res slices
        era5 = self.g_era5[t, :, y0:y1, x0:x1]
        s1 = self.g_s1[m, :, y0:y1, x0:x1]
        s2 = self.g_s2[m, :, y0:y1, x0:x1]
        dem = self.g_dem[0, :, y0:y1, x0:x1]
        world = self.g_world[0, :, y0:y1, x0:x1]
        dyn = self.g_dyn[0, :, y0:y1, x0:x1]

        # target (first landsat band)
        y = self.g_landsat[t, 0, y0:y1, x0:x1]

        # low-res -> high-res tile via index mapping
        y_idx_m = y_map_modis[y0:y1]
        x_idx_m = x_map_modis[x0:x1]
        y_idx_v = y_map_viirs[y0:y1]
        x_idx_v = x_map_viirs[x0:x1]

        modis_lr = self.g_modis[t, :, :, :]
        viirs_lr = self.g_viirs[t, :, :, :]
        modis = modis_lr[:, y_idx_m][:, :, x_idx_m]
        viirs = viirs_lr[:, y_idx_v][:, :, x_idx_v]

        x = np.concatenate([era5, modis, viirs, s1, s2, dem, world, dyn], axis=0)
        return torch.from_numpy(x).float(), torch.from_numpy(y).float()


In [3]:
# 70/10/20 split by samples
samples_per_epoch = 200
n_train = int(samples_per_epoch * 0.7)
n_val = int(samples_per_epoch * 0.1)
n_test = samples_per_epoch - n_train - n_val

train_ds = TileDataset(n_train, seed=1)
val_ds = TileDataset(n_val, seed=2)
test_ds = TileDataset(n_test, seed=3)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=False)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [4]:
import torch.nn.functional as F

TARGET_SCALE = 10000.0


def fill_nan_nearest(x):
    if torch.isfinite(x).all():
        return x
    # nearest-neighbor style resample to fill NaNs/Infs
    x0 = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    h, w = x.shape[-2:]
    x_low = F.interpolate(x0, scale_factor=0.5, mode="nearest")
    x_up = F.interpolate(x_low, size=(h, w), mode="nearest")
    return torch.where(torch.isfinite(x), x, x_up)


def ensure_nchw(x):
    if x.dim() == 4 and x.shape[1] > x.shape[-1]:
        return x.permute(0, 3, 1, 2).contiguous()
    return x


def normalize_batch(x):
    finite = torch.isfinite(x)
    x0 = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    # per-channel stats over batch and spatial dims
    denom = finite.sum(dim=(0, 2, 3)).clamp(min=1)
    mean = (x0 * finite).sum(dim=(0, 2, 3)) / denom
    var = ((x0 - mean[None, :, None, None]) ** 2 * finite).sum(dim=(0, 2, 3)) / denom
    std = var.sqrt().clamp(min=1e-6)
    return (x0 - mean[None, :, None, None]) / std


class SimpleCNN(nn.Module):
    def __init__(self, in_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=1),
        )

    def forward(self, x):
        return self.net(x)


sample_x = next(iter(train_loader))[0]
if sample_x.dim() == 4 and sample_x.shape[1] > sample_x.shape[-1]:
    in_ch = sample_x.shape[-1]
else:
    in_ch = sample_x.shape[1]
model = SimpleCNN(in_ch=in_ch).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

max_epochs = 50

for epoch in range(1, max_epochs + 1):
    model.train()
    train_losses = []
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        xb = ensure_nchw(xb)

        # fill NaNs/Infs in inputs using nearest-neighbor resampling
        if not torch.isfinite(xb).all():
            xb = fill_nan_nearest(xb)
        xb = normalize_batch(xb)

        finite_tgt = torch.isfinite(yb)
        if not finite_tgt.any():
            continue
        yb = torch.nan_to_num(yb, nan=0.0, posinf=0.0, neginf=0.0)
        yb = yb / TARGET_SCALE

        opt.zero_grad(set_to_none=True)
        pred = model(xb).squeeze(1)
        loss = ((pred - yb)[finite_tgt]).pow(2).mean()
        if not torch.isfinite(loss):
            continue
        loss.backward()
        opt.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            xb = ensure_nchw(xb)

            if not torch.isfinite(xb).all():
                xb = fill_nan_nearest(xb)
            xb = normalize_batch(xb)

            finite_tgt = torch.isfinite(yb)
            if not finite_tgt.any():
                continue
            yb = torch.nan_to_num(yb, nan=0.0, posinf=0.0, neginf=0.0)
            yb = yb / TARGET_SCALE

            pred = model(xb).squeeze(1)
            loss = ((pred - yb)[finite_tgt]).pow(2).mean()
            if not torch.isfinite(loss):
                continue
            val_losses.append(loss.item())

    train_loss = float(np.mean(train_losses)) if train_losses else float("nan")
    val_loss = float(np.mean(val_losses)) if val_losses else float("nan")
    print(f"epoch={epoch} train_loss={train_loss:.6f} val_loss={val_loss:.6f}")


RuntimeError: The size of tensor a (256) must match the size of tensor b (46) at non-singleton dimension 3